# Tutorial 6 — Catalogue to prompt, joint GW, and afterglow detections

Start from the **full, unselected structured-jet catalogue** in Tutorial 2.1.
This notebook applies the GRINTA/TED, Crystal Eye and THESEUS/XGIS sensitivities,
adds BNS parameters, computes GWFish SNRs and sky localizations, and models X-ray
follow-up with VegasAfterglow. The same redshift, viewing angle and intrinsic
$E_{\rm iso}$ follow each source through the analysis.

**Run:** select your MAGGPY Python 3.10/3.11 kernel, edit the configuration below,
and Run All. Keep `forecast_pipeline.py` beside this notebook in `Tutorials/`.
CSV tables, event catalogues, figures and run metadata are saved in `Output_files/tutorial6_forecasts/`.
Long GW and afterglow calculations resume from their caches when rerun.

In an environment with MAGGPY installed, the additional packages are:

```bash
python -m pip install VegasAfterglow==2.0.4 joblib PyYAML lalsuite matplotlib ipykernel
python -m pip install "GWFish @ git+https://github.com/janosch314/GWFish.git@5a19d3bf458b8e56aeee619484bff9d0204fcbee"
```

The GWFish source revision above is the one used to check this notebook.
If your working GWFish installation is already available, you can keep it.

In [ ]:
from pathlib import Path
from dataclasses import asdict
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / 'Tutorials' / 'forecast_pipeline.py').is_file() else HERE.parent
if not (ROOT / 'Tutorials' / 'forecast_pipeline.py').is_file():
    raise FileNotFoundError('Open the notebook from the MAGGPY root or its Tutorials directory.')
sys.path.insert(0, str(ROOT / 'Tutorials'))
import forecast_pipeline as fp

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 100)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.2})

## 1. Configuration

Set the catalogue path, generation duration and **the actual $f_j$ used to generate
it**. The fiducial forecast uses $f_j=0.7$. Raw `generate_catalogue` column names and
the renamed Tutorial 2.1 columns are both accepted. Save the full export before
later detected-only examples; it needs `E_iso`, `t_peak`, redshift and viewing angle.

`STRUCTURE_CSV` must match the structure used for that catalogue. The default is
Tutorial 2.1's `struct_results_100.csv`. For Tutorial 2's built-in structure, set it
to `None`; the matching `alpha.txt` and `alpha_e.txt` are then used. If the CSV already
contains `alpha_e` and `alpha_n`, those values take precedence.

Prompt selection uses the supplied flux cuts. `MAX_T90_S=None` keeps every duration;
set it to `2.0` if you want the observer-frame $T_{90}<2$ s selection.
XGIS uses its approximately 2 sr imaging field and an editable duty assumption of 0.65.

In [ ]:
CATALOGUE_PATH = ROOT / 'Output_files' / 'tutorial2_structured' / 'simulated_catalogue.csv'
CATALOGUE_YEARS = 10.0       # N_YEARS in the full-catalogue generation cell
CATALOGUE_FJ = 0.7          # initial_gen[-1] in that run; replace if different
TARGET_FJ = 0.7             # successful-jet fraction for this fiducial forecast
CATALOGUE_REFERENCE_EFFICIENCY = 0.6
STRUCTURE_CSV = ROOT / 'datafiles' / 'structure_constants' / 'struct_results_100.csv'
DEFAULT_STRUCTURE_DIR = ROOT / 'datafiles' / 'structure_constants'
MAX_T90_S = None
SPECTRUM = dict(alpha=-0.67, beta_s=-2.59, n=2.0)
SEED = 42

# These are absolute sky coverage × observing efficiency values.
GBM_EFFICIENCY = 0.6
XGIS_SKY_FRACTION = 0.16
THESEUS_DUTY = 0.65          # editable observing-duty assumption
PROMPT = (
    fp.PromptInstrument('GRINTA_TED', (50., 300.), 1., 0.35, 'ph', GBM_EFFICIENCY),
    fp.PromptInstrument('Crystal_Eye', (50., 300.), 2., 0.455, 'ph', GBM_EFFICIENCY / 2),
    fp.PromptInstrument('THESEUS_XGIS', (30., 150.), 1., 3e-8, 'erg',
                        XGIS_SKY_FRACTION * THESEUS_DUTY, position_arcmin=15., position_fraction=0.9),
)

NETWORKS = {
    'ET_triangle': ['ET'],
    'ET_2L': ['ETS_15', 'ETN_15'],
    'Asharp': ['LLO_sharp', 'LHO_sharp', 'LIO_sharp'],
    'ET_CE': ['ET', 'CE40'],
}
GW_CONFIG_TEMPLATE = ROOT / 'Tutorials' / 'configs' / 'coba.yaml'
PSD_DIR = ROOT / 'Tutorials' / 'psds'
WAVEFORM = 'IMRPhenomD_NRTidalv2'   # tutorial choice; 'TaylorF2' is also available
GW_SNR_THRESHOLD = 8.0
GW_USE_DUTY_CYCLE = False           # same all-operational comparison as Tutorials 3/4
MASS_MEAN, MASS_SIGMA = 1.33, 0.09  # source-frame solar masses, if absent from CSV
LOCALIZATION_CUTS_DEG2 = (1., 10., 100., 1000.)

RUN_AFTERGLOW = True
AFTERGLOW_IMAGERS = fp.DEFAULT_AFTERGLOW
RESPONSE_DELAYS_S = (0., 100., 300., 3600.)
GW_FOLLOWUP_AREA90_MAX_DEG2 = 100.
AFTERGLOW_THETA_CORE_DEG = 5.
AFTERGLOW_N_TIME = 256
AFTERGLOW_RESOLUTIONS = (0.05, 0.2, 5)
AFTERGLOW_EXPOSURE_SAMPLES = 32

N_JOBS = min(4, os.cpu_count() or 1)
GW_CHUNK_SIZE = 256
OUTPUT_DIR = ROOT / 'Output_files' / 'tutorial6_forecasts'
MISSION_YEARS = 3.45                # optional annual-rate × mission-duration table

## 2. Catalogue and rate normalization

Tutorial 2.1's full generator samples viewing angles and redshifts and already
includes $f_j$ and a reference efficiency of 0.6 in its number of sources.
For instrument-specific observability, those rows represent

$$T_{\rm represented} = T_{\rm catalogue}\,0.6\,
   \frac{f_{j,\rm catalogue}}{f_{j,\rm target}}.$$

Apply each instrument's absolute observing efficiency once and report
$N_{\rm detected}/T_{\rm represented}$. For equal $f_j$, this reproduces the GBM-like
normalization and gives Crystal Eye half its coverage. No per-event weights are used.
All rates refer to successful jets within the catalogue's sampled viewing-angle range.

In [ ]:
if not CATALOGUE_PATH.is_file():
    raise FileNotFoundError(f'Set CATALOGUE_PATH to your full Tutorial 2.1 export: {CATALOGUE_PATH}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUTPUT_DIR / 'cache'
cache_settings = (CATALOGUE_YEARS, CATALOGUE_FJ, TARGET_FJ, CATALOGUE_REFERENCE_EFFICIENCY)
EXPOSURE_YEARS = fp.catalogue_exposure(*cache_settings)
sources = fp.canonical_catalogue(pd.read_csv(CATALOGUE_PATH))
sources = fp.attach_temporal_slopes(
    sources, structure_csv=STRUCTURE_CSV, default_structure_dir=DEFAULT_STRUCTURE_DIR)
print(f'{len(sources):,} source rows; {EXPOSURE_YEARS:g} represented all-sky years')
print(f'Viewing angles: {sources.theta_v_deg.min():.3g}–{sources.theta_v_deg.max():.3g} deg')
print(f'f_j: catalogue {CATALOGUE_FJ:g}; forecast {TARGET_FJ:g}')
display(sources[['event_id', 'z', 'theta_v_deg', 'Ep_keV', 'peak_ph_50_300',
                 't_peak_s', 'T90_s', 'E_iso_onaxis_erg']].head())
display(pd.DataFrame([asdict(i) for i in PROMPT]))

## 3. Prompt detections

The catalogue peak photon flux normalizes MAGGPY's smoothly broken power-law
spectrum. Its temporal profile and angle-dependent spectral evolution give
the maximum **window-averaged** flux:

$$\overline F_{\Delta t}=\max_{t_0\ge0}\frac1{\Delta t}
   \int_{t_0}^{t_0+\Delta t}F_{[E_1,E_2]}(t)\,dt.$$

TED and Crystal Eye use photon flux; XGIS uses energy flux integrated over
30–150 keV. The thresholds remain the specified 1 s / 2 s values. An observability
draw accounts for each instrument's sky coverage and efficiency. The saved
trigger time is the end of its best prompt averaging window.

In [ ]:
events = fp.select_prompt(sources, instruments=PROMPT, seed=SEED,
                          max_t90_s=MAX_T90_S, spectral_parameters=SPECTRUM)
prompt_table = fp.prompt_summary(events, EXPOSURE_YEARS, instruments=PROMPT)
display(prompt_table)
prompt_table.to_csv(OUTPUT_DIR / 'prompt_rates.csv', index=False)
events.to_csv(OUTPUT_DIR / 'catalogue_prompt.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
bins = {'z': np.linspace(0, events.z.max(), 25),
        'theta_v_deg': np.linspace(0, max(events.theta_v_deg.max(), 0.1), 25)}
for instrument in PROMPT:
    selected = events[f'prompt_{instrument.name}']
    if selected.any():
        for ax, column in zip(axes, ('z', 'theta_v_deg')):
            counts, edges = np.histogram(events.loc[selected, column], bins=bins[column])
            ax.stairs(counts / EXPOSURE_YEARS, edges, label=instrument.name)
axes[0].set(xlabel='Redshift', ylabel='Detections / year / bin')
axes[1].set(xlabel='Viewing angle [deg]', ylabel='Detections / year / bin')
if any(events[f'prompt_{i.name}'].any() for i in PROMPT):
    axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'prompt_distributions.pdf')
plt.show()

## 4. BNS parameters and GWFish

Existing source masses and GW parameters are reused when present. Otherwise
the tutorial BNS mass distribution and isotropic sky positions are drawn once.
The orbital inclination uses the catalogue viewing angle, with the two jet
hemispheres restored. Planck18 supplies the luminosity distance from the same redshift.
Optional `lambda_1` and `lambda_2` columns are passed through for an EOS-dependent run.

GWFish evaluates SNRs for all catalogue sources. Fisher matrices are calculated
for GW detections, including sources without a prompt trigger, so these can
contribute to the additional-afterglow sample. The explicit nine-parameter Fisher
calculation varies masses, distance, inclination, sky position, polarization,
phase and coalescence time. Redshift is held at the simulated source value.

In [ ]:
binaries = fp.binary_parameters(events, seed=SEED, mass_mean=MASS_MEAN, mass_sigma=MASS_SIGMA)
detector_names = sorted({detector for network in NETWORKS.values() for detector in network})
gw_config = fp.resolve_gw_config(GW_CONFIG_TEMPLATE, PSD_DIR,
                                  OUTPUT_DIR / 'detectors_resolved.yaml', detector_names)
gw_results = fp.run_gw(
    binaries, NETWORKS, gw_config, CACHE_DIR / 'gw', seed=SEED,
    waveform=WAVEFORM, threshold=GW_SNR_THRESHOLD, n_jobs=N_JOBS,
    chunk_size=GW_CHUNK_SIZE, use_duty_cycle=GW_USE_DUTY_CYCLE)
binary_export = binaries.copy()
binary_export.insert(0, 'event_id', events.event_id.to_numpy())
binary_export.to_csv(OUTPUT_DIR / 'binary_parameters.csv', index=False)
gw_export = gw_results.copy()
gw_export.insert(0, 'event_id', events.event_id.to_numpy())
gw_export.to_csv(OUTPUT_DIR / 'gw_events.csv', index=False)

gw_rows = []
for network in NETWORKS:
    detected = gw_results[f'gw_{network}']
    area = gw_results[f'area90_{network}_deg2']
    gw_rows.append({'network': network, 'sample': 'successful jets within catalogue angles',
                    **fp.rate_fields(detected.sum(), EXPOSURE_YEARS, len(events)),
                    'valid_localizations': int((detected & area.notna() & (area > 0)).sum())})
gw_table = pd.DataFrame(gw_rows)
display(gw_table)
gw_table.to_csv(OUTPUT_DIR / 'gw_catalogue_rates.csv', index=False)

## 5. Joint prompt + GW rates and localizations

A joint event passes that instrument's prompt threshold and observability selection,
and the network SNR threshold. GW localization areas are reported at **90%** in deg².
Invalid or unconstrained areas are left missing and excluded from localization cuts.
Yellow Book XGIS imaging requirements are ≤7 arcmin for 50% of triggered sGRBs
and ≤15 arcmin for 90% (both at 90% confidence). The 15 arcmin / 0.9 benchmark is
listed separately; this notebook computes GW areas and does not fit per-event XGIS positions.

`mc_low90_per_yr` and `mc_high90_per_yr` are exact binomial intervals for the
finite-catalogue detection fraction, converted to annual rates. They measure Monte
Carlo sampling uncertainty; population-parameter uncertainty is separate.

In [ ]:
joint_table = fp.joint_summary(events, gw_results, EXPOSURE_YEARS,
                                instruments=PROMPT, localization_cuts=LOCALIZATION_CUTS_DEG2)
display(joint_table)
joint_table.to_csv(OUTPUT_DIR / 'joint_prompt_rates.csv', index=False)

joint_table.pivot(index='network', columns='instrument', values='rate_per_yr').plot.bar(
    figsize=(11, 4), rot=0, ylabel='Prompt + GW detections / year')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'joint_prompt_rates.pdf')
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for instrument in PROMPT:
    for network in NETWORKS:
        joint = events[f'prompt_{instrument.name}'] & gw_results[f'gw_{network}']
        area = gw_results.loc[joint, f'area90_{network}_deg2'].to_numpy()
        area = np.sort(area[np.isfinite(area) & (area > 0)])
        if len(area):
            ax.step(area, np.arange(1, len(area) + 1) / EXPOSURE_YEARS,
                    where='post', label=f'{instrument.name} + {network}')
for cut in (1, 10, 100):
    ax.axvline(cut, color='grey', lw=0.8, ls=':')
ax.set(xscale='log', xlabel='90% GW sky area [deg²]', ylabel='Joint detections / year below area')
if ax.get_legend_handles_labels()[0]:
    ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'joint_localization_cumulative.pdf')
plt.show()

## 6. X-ray afterglows

VegasAfterglow uses the catalogue's on-axis rest-frame $E_{\rm iso}$, redshift and
viewing angle. The supplied afterglow setup is retained: a 5° top-hat jet without
spreading, $\epsilon_e=0.1$, $p=2.2$, $\xi_e=1$,
$n_{\rm ISM}\sim U(2.5\times10^{-4},1.5\times10^{-2})$ cm$^{-3}$,
$\log_{10}\epsilon_B\sim U(-4,-2)$ and $\log_{10}\Gamma_0\sim\mathcal N(2.3,0.3^2)$.
These parameters are drawn once per source and shared across every comparison.

| Imager | Band | Exposure(s) | Flux sensitivity |
|---|---|---|---|
| GRINTA HXI | 5–30 keV | 10, 100, 1,000, 10,000, 100,000 s | $1.2\times10^{-3}\sqrt{10^4/\Delta t}$ ph cm$^{-2}$ s$^{-1}$ |
| THESEUS SXI | 0.3–5 keV | 100 / 1,500 s | $10^{-10}$ / $1.8\times10^{-11}$ erg cm$^{-2}$ s$^{-1}$ (3σ) |

HXI follows the supplied analysis; SXI uses Table 3-1 of the
[THESEUS Yellow Book](https://sci.esa.int/documents/34375/36249/Theseus_YB_final.pdf).
The quoted SXI imaging position is taken as 2 arcmin. Exposure-averaged flux,
integrated over the imager's band, is compared with its matching exposure sensitivity.

For each response delay, the scan finds the best exposure that starts after the
alert plus that delay, over 1–$10^7$ s. Prompt follow-up starts after the prompt
trigger. GW follow-up starts after coalescence and requires the configured GW
localization cut. `followup_fraction=1` reports the yield conditional on arranging
that follow-up; edit each imager's value to include its availability. Cross-instrument
rows likewise assume follow-up is arranged.

In [ ]:
display(pd.DataFrame([asdict(i) for i in AFTERGLOW_IMAGERS]))
microphysics = fp.afterglow_parameters(events, seed=SEED)
micro_export = microphysics.copy()
micro_export.insert(0, 'event_id', events.event_id.to_numpy())
micro_export.to_csv(OUTPUT_DIR / 'afterglow_parameters.csv', index=False)

if RUN_AFTERGLOW:
    afterglow_events = fp.run_afterglow(
        events, gw_results, microphysics, CACHE_DIR / 'afterglow',
        instruments=AFTERGLOW_IMAGERS, prompt_instruments=PROMPT,
        delays=RESPONSE_DELAYS_S, localization_cut=GW_FOLLOWUP_AREA90_MAX_DEG2,
        theta_core_deg=AFTERGLOW_THETA_CORE_DEG, n_jobs=N_JOBS,
        n_time=AFTERGLOW_N_TIME, resolutions=AFTERGLOW_RESOLUTIONS,
        exposure_samples=AFTERGLOW_EXPOSURE_SAMPLES)
    afterglow_events.to_csv(OUTPUT_DIR / 'afterglow_events.csv', index=False)
    afterglow_table = fp.afterglow_summary(
        events, gw_results, afterglow_events, EXPOSURE_YEARS,
        instruments=AFTERGLOW_IMAGERS, prompt_instruments=PROMPT,
        delays=RESPONSE_DELAYS_S, localization_cut=GW_FOLLOWUP_AREA90_MAX_DEG2, seed=SEED)
    afterglow_table.to_csv(OUTPUT_DIR / 'afterglow_rates.csv', index=False)
    display(afterglow_table)
else:
    afterglow_events = pd.DataFrame(columns=fp.AF_COLUMNS)
    afterglow_table = pd.DataFrame()
    print('Afterglow disabled; prompt and GW tables have been saved.')

The afterglow rate table reports these distinct counts for each prompt instrument,
imager, GW network and response delay:

- `prompt_afterglow_joint`: prompt + GW detections with a detectable afterglow after the prompt alert.
- `gw_triggered_afterglow`: localized GW detections with a detectable afterglow after the GW alert.
- `extra_without_prompt`: the subset of the latter without a prompt detection in that instrument.
- `unique_prompt_or_added_afterglow`: prompt + GW events plus those exclusive additions, counted once.

In [ ]:
if not afterglow_table.empty:
    fig, axes = plt.subplots(1, len(AFTERGLOW_IMAGERS), figsize=(6 * len(AFTERGLOW_IMAGERS), 4), squeeze=False)
    for ax, imager in zip(axes[0], AFTERGLOW_IMAGERS):
        selected = afterglow_table[afterglow_table.imager == imager.name]
        for (prompt_name, network), group in selected.groupby(['prompt_instrument', 'network']):
            group = group.sort_values('delay_s')
            ax.plot(group.delay_s, group.extra_without_prompt_per_yr, 'o-',
                    label=f'{prompt_name} + {network}')
        ax.set(xlabel='Response delay [s]', ylabel='Additional GW + afterglow detections / year', title=imager.name)
        ax.set_xscale('symlog', linthresh=10)
        ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'extra_afterglow_rates.pdf')
    plt.show()

## 7. Compact tables for the chapter and saved run

`chapter_numbers.csv` combines prompt + GW rates, their GW localization counts,
afterglow follow-up and the exclusive additional detections. `mission_expectations.csv`
multiplies annual rates by `MISSION_YEARS`; its values are expected event numbers.
Per-event CSVs retain the fluxes, observability flags, SNRs, best afterglow exposures
and detection margins for further cuts. The full settings and package versions
accompany the tables in `run_metadata.json`.

In [ ]:
chapter_numbers = joint_table.rename(columns={'instrument': 'prompt_instrument',
                                                'count': 'prompt_joint_count',
                                                'rate_per_yr': 'prompt_joint_per_yr'})
if not afterglow_table.empty:
    chapter_numbers = chapter_numbers.merge(
        afterglow_table.drop(columns='prompt_joint_count'),
        on=['prompt_instrument', 'network'], how='left', validate='one_to_many')
chapter_numbers.to_csv(OUTPUT_DIR / 'chapter_numbers.csv', index=False)
display(chapter_numbers)
mission_expectations = chapter_numbers.copy()
mission_expectations['mission_years'] = MISSION_YEARS
for column in chapter_numbers.columns:
    if column.endswith('_per_yr'):
        mission_expectations[column.removesuffix('_per_yr') + '_expected_mission_count'] = (
            chapter_numbers[column] * MISSION_YEARS)
mission_expectations.to_csv(OUTPUT_DIR / 'mission_expectations.csv', index=False)

settings = {
    'catalogue_years': CATALOGUE_YEARS, 'catalogue_fj': CATALOGUE_FJ, 'target_fj': TARGET_FJ,
    'reference_efficiency': CATALOGUE_REFERENCE_EFFICIENCY,
    'structure_csv': str(STRUCTURE_CSV), 'default_structure_dir': str(DEFAULT_STRUCTURE_DIR),
    'structure_sha256': fp.sha256(STRUCTURE_CSV.read_bytes()).hexdigest() if STRUCTURE_CSV else None,
    'observed_angle_range_deg': [events.theta_v_deg.min(), events.theta_v_deg.max()],
    'spectrum': SPECTRUM, 'max_t90_s': MAX_T90_S, 'seed': SEED,
    'prompt': [asdict(i) for i in PROMPT], 'networks': NETWORKS,
    'gw_waveform': WAVEFORM, 'gw_snr_threshold': GW_SNR_THRESHOLD,
    'gw_use_duty_cycle': GW_USE_DUTY_CYCLE, 'mass_mean': MASS_MEAN, 'mass_sigma': MASS_SIGMA,
    'detectors_resolved_yaml': gw_config.read_text(),
    'psd_sha256': {name: fp.sha256(Path(value['psd_data']).read_bytes()).hexdigest()
                   for name, value in __import__('yaml').safe_load(gw_config.read_text()).items()},
    'localization_cuts_deg2': LOCALIZATION_CUTS_DEG2,
    'run_afterglow': RUN_AFTERGLOW, 'afterglow_imagers': [asdict(i) for i in AFTERGLOW_IMAGERS],
    'response_delays_s': RESPONSE_DELAYS_S, 'gw_followup_area90_max_deg2': GW_FOLLOWUP_AREA90_MAX_DEG2,
    'afterglow_theta_core_deg': AFTERGLOW_THETA_CORE_DEG, 'afterglow_n_time': AFTERGLOW_N_TIME,
    'afterglow_resolutions': AFTERGLOW_RESOLUTIONS, 'afterglow_exposure_samples': AFTERGLOW_EXPOSURE_SAMPLES,
    'mission_years': MISSION_YEARS,
}
fp.save_run_metadata(OUTPUT_DIR / 'run_metadata.json', CATALOGUE_PATH, EXPOSURE_YEARS, settings)
print(f'Saved all tables and figures in {OUTPUT_DIR}')

Source examples: MAGGPY [Tutorial 2.1](tutorial2.1_structured_jet.ipynb),
[Tutorial 3](tutorial3_gwfish_joint_detections.ipynb),
[Tutorial 4](tutorial4_gwfish_skyloc.ipynb),
[GWFish](https://github.com/janosch314/GWFish),
[VegasAfterglow exposure API](https://yihanwangastro.github.io/VegasAfterglow/),
and [THESEUS Yellow Book, Table 3-1](https://sci.esa.int/documents/34375/36249/Theseus_YB_final.pdf).